In [ ]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# get all csv files in the data directory for run of interest
data_dir = Path("output/runs/1/data")
csv_files = list(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files.")

# read all files with mixed types
arrays = []
for i, f in enumerate(csv_files):
    arr = np.genfromtxt(f, delimiter=',', dtype=None, encoding='utf-8')
    arrays.append(arr)
    if (i + 1) % 1000 == 0:
        print(f"Loaded {i+1}/{len(csv_files)} files")

# stack all arrays vertically
data = np.concatenate(arrays)
# convert data to pandas df
df = pd.DataFrame(data)
# remove all rows that are exact duplicates, since they are removing the rows of column names
df = df.drop_duplicates()
# make first row the header
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

In [ ]:
df

In [ ]:
# get distribution of accuracy via histogram
plt.figure(figsize=(10, 6))
plt.hist(df['test_acc'].astype(float), bins=10, edgecolor='black', alpha=0.7)
plt.title('Distribution of Test Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Test Accuracy', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# get the different unique values for mv_dim_missing_strategy
unique_strategies = df['mv_dim_missing_strategy'].unique()
print("Unique mv_dim_missing_strategy values:", unique_strategies)

# Comprehensive Variability Analysis: All Decision Dimensions

We analyze the inter-bucket and intra-bucket variability for each decision in the pipeline:
1. **Spell Gap Threshold** (`mv_dim_spell_gap_threshold`)
2. **Missing Data Strategy** (`mv_dim_missing_strategy`)
3. **Train-Test Split Seed** (`mv_dim_split_seed`)
4. **Model** (`mv_dim_model`)

In [ ]:
# Define all decision dimensions in pipeline order
decision_dims = [
    ("mv_dim_missing_strategy", "Missing Strategy"),
    ("mv_dim_model", "Model"),
    ("mv_dim_new_job_no_notif_threshold", "New Job No-Notification Threshold"),
    ("mv_dim_new_job_notif_threshold", "New Job Notification Threshold"),
    ("mv_dim_spell_gap_threshold", "Spell Gap Threshold"),
    ("mv_dim_split_seed", "Train-Test Split Seed"),
]

# Ensure test_acc is float
df['test_acc'] = df['test_acc'].astype(float)

# Store results for each dimension
variability_results = []

for dim_col, dim_name in decision_dims:
    # Other dimensions (all except current)
    other_dims = [d[0] for d in decision_dims if d[0] != dim_col]
    
    # === Inter-Bucket Variability ===
    # Variance within each bucket (how spread are results for each choice)
    inter_var = df.groupby(dim_col)['test_acc'].var()
    mean_inter_var = inter_var.mean()
    
    # Between-bucket variance (variance of bucket means)
    bucket_means = df.groupby(dim_col)['test_acc'].mean()
    between_bucket_var = bucket_means.var()
    
    # === Intra-Bucket Variability ===
    # Group by all OTHER dimensions, measure variance as this dimension varies
    intra_bucket = df.groupby(other_dims)['test_acc'].agg(['var', 'std', 'count']).reset_index()
    intra_bucket_valid = intra_bucket[intra_bucket['count'] > 1]
    mean_intra_var = intra_bucket_valid['var'].mean() if len(intra_bucket_valid) > 0 else np.nan
    
    # Number of unique choices
    n_choices = df[dim_col].nunique()
    
    variability_results.append({
        'dimension': dim_name,
        'column': dim_col,
        'n_choices': n_choices,
        'mean_inter_bucket_var': mean_inter_var,
        'between_bucket_var': between_bucket_var,
        'mean_intra_bucket_var': mean_intra_var,
    })

# Create summary dataframe
variability_df = pd.DataFrame(variability_results)
variability_df = variability_df.set_index('dimension')
print("Variability Summary by Decision Dimension")
print("=" * 80)
print(variability_df[['n_choices', 'mean_inter_bucket_var', 'between_bucket_var', 'mean_intra_bucket_var']].round(6))

In [ ]:
# Visualization 1: Bar chart comparing variability measures across dimensions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dims = variability_df.index.tolist()
x = np.arange(len(dims))
width = 0.6

# Plot 1: Mean Intra-Bucket Variance (variance attributable to each decision)
ax1 = axes[0]
bars1 = ax1.bar(x, variability_df['mean_intra_bucket_var'], width, color='steelblue', edgecolor='black')
ax1.set_ylabel('Variance', fontsize=11)
ax1.set_title('Intra-Bucket Variance\n(Variance Attributable to Decision)', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(dims, rotation=45, ha='right', fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Between-Bucket Variance (how different are the bucket means)
ax2 = axes[1]
bars2 = ax2.bar(x, variability_df['between_bucket_var'], width, color='darkorange', edgecolor='black')
ax2.set_ylabel('Variance', fontsize=11)
ax2.set_title('Between-Bucket Variance\n(Variance of Choice Means)', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(dims, rotation=45, ha='right', fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Mean Inter-Bucket Variance (spread within each bucket)
ax3 = axes[2]
bars3 = ax3.bar(x, variability_df['mean_inter_bucket_var'], width, color='seagreen', edgecolor='black')
ax3.set_ylabel('Variance', fontsize=11)
ax3.set_title('Mean Inter-Bucket Variance\n(Avg Spread Within Each Choice)', fontsize=12, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(dims, rotation=45, ha='right', fontsize=9)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Ranked bar chart - which decision drives uncertainty most?
fig, ax = plt.subplots(figsize=(10, 6))

# Sort by intra-bucket variance (variance attributable to the decision)
sorted_df = variability_df.sort_values('mean_intra_bucket_var', ascending=True)

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sorted_df)))
y_pos = np.arange(len(sorted_df))

bars = ax.barh(y_pos, sorted_df['mean_intra_bucket_var'], color=colors, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_df.index, fontsize=11)
ax.set_xlabel('Variance Attributable to Decision', fontsize=11)
ax.set_title('Decision Dimensions Ranked by Impact on Accuracy Uncertainty', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(sorted_df.iterrows()):
    ax.text(row['mean_intra_bucket_var'] + 0.0001, i, f"{row['mean_intra_bucket_var']:.6f}", 
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("INTERPRETATION: Decision dimensions ranked from LEAST to MOST impact on accuracy")
print("=" * 80)
for i, (idx, row) in enumerate(sorted_df.iterrows(), 1):
    print(f"{i}. {idx}: Intra-bucket variance = {row['mean_intra_bucket_var']:.6f}")

In [ ]:
# Visualization 3: Boxplots for each decision dimension
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.flatten()

for i, (dim_col, dim_name) in enumerate(decision_dims):
    ax = axes[i]
    
    # Get unique values and sort them
    unique_vals = sorted(df[dim_col].unique(), key=lambda x: str(x))
    
    # Create boxplot data
    box_data = [df[df[dim_col] == val]['test_acc'].values for val in unique_vals]
    
    bp = ax.boxplot(box_data, patch_artist=True)
    
    # Color the boxes
    colors = plt.cm.Set3(np.linspace(0, 1, len(unique_vals)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    ax.set_xticklabels([str(v)[:15] for v in unique_vals], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Test Accuracy', fontsize=10)
    ax.set_title(f'{dim_name}', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Test Accuracy Distribution by Each Decision Dimension', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary Table: Detailed statistics for each decision dimension
print("=" * 100)
print("DETAILED SUMMARY TABLE: Variability by Decision Dimension (Pipeline Order)")
print("=" * 100)

summary_data = []
for dim_col, dim_name in decision_dims:
    unique_vals = df[dim_col].unique()
    
    # Stats per choice
    group_stats = df.groupby(dim_col)['test_acc'].agg(['mean', 'std', 'count'])
    
    summary_data.append({
        'Decision': dim_name,
        'N Choices': len(unique_vals),
        'Overall Mean': df['test_acc'].mean(),
        'Overall Std': df['test_acc'].std(),
        'Min Choice Mean': group_stats['mean'].min(),
        'Max Choice Mean': group_stats['mean'].max(),
        'Range of Means': group_stats['mean'].max() - group_stats['mean'].min(),
        'Between-Choice Var': variability_df.loc[dim_name, 'between_bucket_var'],
        'Intra-Bucket Var': variability_df.loc[dim_name, 'mean_intra_bucket_var'],
    })

summary_table = pd.DataFrame(summary_data)
summary_table = summary_table.set_index('Decision')
print(summary_table.round(6).to_string())

print("\n" + "=" * 100)
print("KEY METRIC: 'Intra-Bucket Var' = Variance attributable to each decision alone")
print("Higher values indicate the decision has MORE impact on accuracy uncertainty")
print("=" * 100)

In [ ]:
# Visualization 4: Heatmap of mean accuracy by pairs of decisions
# Pick the top 2 most impactful decisions for a 2D heatmap

top_2 = variability_df.sort_values('mean_intra_bucket_var', ascending=False).head(2)
dim1_col = top_2.iloc[0]['column']
dim1_name = top_2.index[0]
dim2_col = top_2.iloc[1]['column']
dim2_name = top_2.index[1]

# Create pivot table
pivot = df.pivot_table(values='test_acc', index=dim1_col, columns=dim2_col, aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')

# Labels
ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_xticklabels([str(c)[:20] for c in pivot.columns], rotation=45, ha='right', fontsize=9)
ax.set_yticklabels([str(r)[:20] for r in pivot.index], fontsize=9)

ax.set_xlabel(dim2_name, fontsize=12)
ax.set_ylabel(dim1_name, fontsize=12)
ax.set_title(f'Mean Test Accuracy: {dim1_name} × {dim2_name}\n(Top 2 Most Impactful Decisions)', 
             fontsize=13, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Mean Test Accuracy', fontsize=11)

# Add text annotations
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.iloc[i, j]
        if not np.isnan(val):
            text_color = 'white' if val < pivot.values.mean() else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', color=text_color, fontsize=8)

plt.tight_layout()
plt.show()